# 03 · Internal Evaluation: Ensemble, MC Dropout and Calibration

Evaluates the 15 trained models on the lesion-grouped test folds of HAM10000:

* single backbones versus the soft-voting ensemble;
* **MC Dropout** on the classification heads (30 passes per model, 90 samples per image) and **temperature scaling** fitted on each
  fold's validation set;
* balanced accuracy (primary metric), accuracy, macro F1, macro ROC-AUC, (pre)malignant-vs-benign AUC, ECE, Brier score, NLL;
* pooled out-of-fold confusion matrix, per-class metrics, reliability diagrams and the choice of uncertainty signal.

Features are extracted once per checkpoint and cached; every later notebook rebuilds the framework from these caches.

**Inputs:** notebook 01 outputs, checkpoints · **Outputs:** `results/internal_results.csv`, `results/confusion_matrix.png`

## Setup

Paths can be edited in the next cell or set as environment variables (`HAM_PROJECT_OUT`, `HAM10000_DIR`, `ISIC2019_DIR`).

In [ ]:
import os, json, math, time, random, platform, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                                   # progress bars are optional
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

In [ ]:
# ---------------------------------------------------------------- paths (edit here or set the environment variables)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

def env_path(name, default):
    return Path(os.environ.get(name, str(default))).expanduser()

PROJECT_OUT  = env_path('HAM_PROJECT_OUT', r'D:/HAM_PROJECT/outputs_v2')   # caches, checkpoints, features, tables (not in Git)
HAM_DIR      = env_path('HAM10000_DIR', r'D:/HAM_PROJECT/dataset')      # HAM10000 images, metadata and lesion masks
ISIC2019_DIR = env_path('ISIC2019_DIR', r'D:/Skin Dataset/ISIC2019')    # ISIC 2019 training images, ground truth, metadata
RESULTS_DIR  = env_path('RESULTS_DIR', REPO / 'results')             # tables and figures shown in the README
SPLITS_DIR   = env_path('SPLITS_DIR', REPO / 'splits')               # published fold assignment

# ---------------------------------------------------------------- study settings (keep unchanged to reproduce the results)
CLASSES   = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
MALIGNANT = ['akiec', 'bcc', 'mel']                                  # (pre)malignant classes
MODELS    = ['efficientnet_b4', 'convnext_base', 'densenet121']
LABEL_OF  = {'efficientnet_b4': 'EfficientNet-B4', 'convnext_base': 'ConvNeXt-Base', 'densenet121': 'DenseNet-121'}
IMG_SIZE, SEED, N_FOLDS = 224, 42, 5
DROPOUT, MC_SAMPLES = 0.4, 30                                       # head dropout; MC Dropout passes per model
UNC_QUANTILE, GMM_N, N_NVREF = 0.75, 3, 1000                        # uncertainty flag; HBDS levels; nevus reference size
VARIANT = 'focal_ls'                                                # main models (focal loss + label smoothing)
EXT_MC_SEED0 = 555                                                  # MC seed offset for external data

C = len(CLASSES)
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
MAL_IDX = [CLASS2IDX[c] for c in MALIGNANT]
EPS = 1e-12
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DIRS = {k: PROJECT_OUT / k for k in ['splits', 'checkpoints', 'cache', 'tables', 'figures']}
for _d in list(DIRS.values()) + [RESULTS_DIR, SPLITS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def mean_sd(values, digits=4):
    s = pd.Series(values, dtype=float).dropna()
    return f'{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}' if len(s) > 1 else (f'{s.mean():.{digits}f}' if len(s) else 'n/a')

print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | device: {DEVICE}')
print(f'Project outputs: {PROJECT_OUT}')

In [ ]:
import cv2
from PIL import Image

def find_file(root, name):
    """Locate a dataset file by name anywhere below root."""
    direct = Path(root) / name
    if direct.is_file():
        return direct
    hits = sorted(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under {root}. See the Dataset section of README.md.")
    return hits[0]

def image_index(root, exclude=('segmentation', 'mask')):
    """Map image stem -> path for all images below root (lesion masks excluded)."""
    index = {}
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and not any(e in p.as_posix().lower() for e in exclude):
            index.setdefault(p.stem, p)
    return index

def read_rgb(path):
    with Image.open(path) as im:
        return np.array(im.convert('RGB'))

def read_rgb_224(path):
    """Pipeline image loading: PIL -> RGB -> cv2 bilinear resize to 224 x 224."""
    return cv2.resize(read_rgb(path), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def build_image_cache(paths, npy_path, json_path, desc='caching images'):
    """Decode every image once into a uint8 array (N, 224, 224, 3). Reused if it already exists."""
    paths = [str(p) for p in paths]
    if Path(npy_path).exists() and Path(json_path).exists():
        cached = json.load(open(json_path))
        if [Path(p).stem for p in cached] == [Path(p).stem for p in paths]:
            return np.load(npy_path, mmap_mode='r')
        raise RuntimeError(f'{json_path} exists but lists different images - delete it to rebuild the cache.')
    Path(npy_path).parent.mkdir(parents=True, exist_ok=True)
    arr = np.lib.format.open_memmap(npy_path, mode='w+', dtype=np.uint8, shape=(len(paths), IMG_SIZE, IMG_SIZE, 3))
    for i, p in enumerate(tqdm(paths, desc=desc)):
        arr[i] = read_rgb_224(p)
    arr.flush(); del arr
    json.dump(paths, open(json_path, 'w'))
    return np.load(npy_path, mmap_mode='r')

def load_ham():
    """HAM10000 metadata in the order of the image cache, labels and lesion groups."""
    meta = pd.read_csv(find_file(HAM_DIR, 'HAM10000_metadata.csv')).drop_duplicates('image_id').set_index('image_id')
    order_file = DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.json'
    if not order_file.exists():
        raise FileNotFoundError(f'{order_file} not found - run notebook 01 first.')
    ids = [Path(p).stem for p in json.load(open(order_file))]
    ham = meta.loc[ids].reset_index()
    return ham, ham['dx'].map(CLASS2IDX).astype(int).values, ham['lesion_id'].astype(str).values

def load_splits():
    f = DIRS['splits'] / 'cv_splits.json'
    if not f.exists():
        raise FileNotFoundError(f'{f} not found - run notebook 01 first.')
    return {int(k): {kk: np.asarray(vv, dtype=int) for kk, vv in v.items()} for k, v in json.load(open(f)).items()}

In [ ]:
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

def make_head(in_dim, p=DROPOUT):
    """LayerNorm -> Dropout(0.4) -> Linear(512) -> GELU -> Dropout(0.2) -> Linear(7)."""
    return nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(p), nn.Linear(in_dim, 512),
                         nn.GELU(), nn.Dropout(p / 2), nn.Linear(512, C))

class SkinLesionModel(nn.Module):
    """ImageNet-pretrained timm backbone with global average pooling + classification head."""
    def __init__(self, model_name, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.feat_dim = self.backbone.num_features
        self.classifier = make_head(self.feat_dim)

    def forward_features(self, x):
        return self.backbone(x)

    def forward(self, x):
        return self.classifier(self.backbone(x))

def ckpt_path(k, m, variant=VARIANT):
    return DIRS['checkpoints'] / f'fold{k}' / variant / f'{m}.pth'

def load_model(k, m, variant=VARIANT):
    p = ckpt_path(k, m, variant)
    if not p.exists():
        raise FileNotFoundError(f'Checkpoint {p} not found - train it with notebook 02 or download the released weights.')
    ck = torch.load(p, map_location='cpu', weights_only=True)
    model = SkinLesionModel(m)
    model.load_state_dict(ck['state_dict'] if 'state_dict' in ck else ck, strict=True)
    model.to(DEVICE).eval()
    for q in model.parameters():
        q.requires_grad_(False)
    return model

INFER_BATCH = 64 if DEVICE.type == 'cuda' else 16

@torch.no_grad()
def extract_features(model, images, rows=None, batch_size=INFER_BATCH):
    """Pooled backbone features (eval mode, float16 autocast on GPU), returned as float16 as in the study."""
    rows = np.arange(len(images)) if rows is None else np.asarray(rows)
    out = []
    for s in range(0, len(rows), batch_size):
        x = torch.stack([eval_transform(image=np.ascontiguousarray(images[i]))['image'] for i in rows[s:s + batch_size]]).to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            f = model.forward_features(x)
        out.append(f.float().cpu().numpy().astype(np.float16))
    return np.concatenate(out) if out else np.zeros((0, model.feat_dim), np.float16)

In [ ]:
from scipy.optimize import minimize_scalar
from sklearn.covariance import LedoitWolf
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

def fold_dir(k, variant=VARIANT):
    return DIRS['cache'] / f'fold{k}' / variant

def feat_path(k, m, variant=VARIANT):
    return fold_dir(k, variant) / f'{m}_features.npz'

def head_path(k, m, variant=VARIANT):
    return fold_dir(k, variant) / f'{m}_head.pt'

def ensure_fold_cache(k, images, labels, splits, variant=VARIANT):
    """Save each model's head and its validation, test and nevus-reference features (skipped if already cached)."""
    for m in MODELS:
        if feat_path(k, m, variant).exists() and head_path(k, m, variant).exists():
            continue
        fold_dir(k, variant).mkdir(parents=True, exist_ok=True)
        model = load_model(k, m, variant)
        torch.save({'in_dim': model.feat_dim, 'state': model.classifier.state_dict()}, head_path(k, m, variant))
        sp = splits[k]
        nv_train = sp['train'][labels[sp['train']] == CLASS2IDX['nv']]
        nvref_idx = np.sort(np.random.default_rng(SEED + k).choice(nv_train, size=min(N_NVREF, len(nv_train)), replace=False))
        np.savez(feat_path(k, m, variant), val=extract_features(model, images, sp['val']),
                 test=extract_features(model, images, sp['test']), nvref=extract_features(model, images, nvref_idx),
                 nvref_idx=nvref_idx)
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

def load_head(k, m, variant=VARIANT):
    d = torch.load(head_path(k, m, variant), map_location='cpu', weights_only=True)
    h = make_head(d['in_dim']); h.load_state_dict(d['state'])
    return h.to(DEVICE).eval()

def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

@torch.no_grad()
def head_logits(head, feats, chunk=8192):
    head.eval(); out = []
    for i in range(0, len(feats), chunk):
        out.append(head(torch.from_numpy(np.asarray(feats[i:i + chunk], np.float32)).to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def head_mc(head, feats, n, seed, chunk=8192):
    """MC Dropout on the head only (the backbones contain no dropout, so this equals full stochastic passes)."""
    head.train(); torch.manual_seed(seed)
    X = torch.from_numpy(np.asarray(feats, np.float32)).to(DEVICE)
    out = np.empty((n, len(feats), C), np.float32)
    for s in range(n):
        for i in range(0, len(feats), chunk):
            out[s, i:i + chunk] = F.softmax(head(X[i:i + chunk]), 1).cpu().numpy()
    head.eval()
    return out

def mc_stats(samples):
    pbar = samples.mean(0); chat = pbar.argmax(1)
    var = samples[:, np.arange(samples.shape[1]), chat].var(0)            # variance of the predicted-class probability
    ent = -(pbar * np.log(pbar + EPS)).sum(1)
    exp_ent = -(samples * np.log(samples + EPS)).sum(2).mean(0)
    return {'probs': pbar, 'var': var, 'entropy': ent, 'mi': ent - exp_ent}

def nll(p, y):
    return float(-np.mean(np.log(p[np.arange(len(y)), y] + EPS)))

def fit_temperature(logits, y):
    lg = np.asarray(logits, np.float64)
    return float(minimize_scalar(lambda T: nll(softmax_np(lg / T), y), bounds=(0.05, 10.0), method='bounded').x)

def mahalanobis(X, mu, prec, chunk=4096):
    out = np.empty(len(X))
    for i in range(0, len(X), chunk):
        d = np.asarray(X[i:i + chunk], np.float64) - mu
        out[i:i + chunk] = np.sqrt(np.maximum(np.sum((d @ prec) * d, 1), 0))
    return out

def fit_gmm(scores, k):
    g = GaussianMixture(n_components=k, covariance_type='full', n_init=10, max_iter=300, random_state=SEED).fit(scores.reshape(-1, 1))
    order = np.argsort(g.means_.ravel()); rank = np.empty(k, int); rank[order] = np.arange(k)
    return g, rank

def gmm_levels(g, rank, s):
    return rank[g.predict(np.asarray(s).reshape(-1, 1))]

def _value(v, k):
    return v(k) if callable(v) else v

def build_fold(k, ext=None, variant=VARIANT):
    """Rebuild the full framework of fold k from cached features.

    ext: optional {name: dict(feats=fn(k, m), y=..., ymal=..., groups=...)} for additional (external) image sets.
    Every fitted quantity (temperatures, HBDS reference and scaling, GMM levels, uncertainty threshold) uses the
    HAM10000 validation partition of fold k only.
    """
    ext = ext or {}
    sp = SPLITS[k]; base = SEED * 1000 + k; y_val = LABELS[sp['val']]
    parts = ['val', 'test'] + list(ext)
    feats = {m: {} for m in MODELS}
    for m in MODELS:
        z = np.load(feat_path(k, m, variant))
        feats[m].update(val=z['val'], test=z['test'], nvref=z['nvref'])
        for ds, d in ext.items():
            feats[m][ds] = d['feats'](k, m)
    R = {p: {'single_det': {}, 'single_cal': {}} for p in parts}
    samples = {p: [] for p in parts}; Ts = {}
    for mi, m in enumerate(MODELS):
        h = load_head(k, m, variant)
        lg = {p: head_logits(h, feats[m][p]) for p in parts}
        Ts[m] = fit_temperature(lg['val'], y_val)
        for p in parts:
            R[p]['single_det'][m] = softmax_np(lg[p]); R[p]['single_cal'][m] = softmax_np(lg[p] / Ts[m])
            seed = (base + 17 * mi + (0 if p == 'val' else 7919)) if p in ('val', 'test') else EXT_MC_SEED0 + mi
            samples[p].append(head_mc(h, feats[m][p], MC_SAMPLES, seed))
        del h
    for p in parts:
        R[p]['ens_det'] = np.mean([R[p]['single_det'][m] for m in MODELS], 0)
        R[p]['mc'] = mc_stats(np.concatenate(samples[p], 0)); samples[p] = None
    T_det = fit_temperature(np.log(R['val']['ens_det'] + EPS), y_val)
    T = fit_temperature(np.log(R['val']['mc']['probs'] + EPS), y_val)
    lw = LedoitWolf(assume_centered=False).fit(np.concatenate([feats[m]['nvref'] for m in MODELS], 1).astype(np.float64))
    dist = {p: mahalanobis(np.concatenate([feats[m][p] for m in MODELS], 1), lw.location_, lw.precision_) for p in parts}
    dmin, dmax = dist['val'].min(), dist['val'].max()
    for p in parts:
        R[p]['ens_det_cal'] = softmax_np(np.log(R[p]['ens_det'] + EPS) / T_det)
        R[p]['full'] = softmax_np(np.log(R[p]['mc']['probs'] + EPS) / T)            # full framework prediction
        R[p]['hbds'] = np.clip((dist[p] - dmin) / (dmax - dmin + EPS), 0, 1) * 100
    g, rank = fit_gmm(R['val']['hbds'], GMM_N)
    tau_u = float(np.quantile(R['val']['mc']['var'], UNC_QUANTILE))
    for p in parts:
        R[p]['level'] = gmm_levels(g, rank, R[p]['hbds']); R[p]['unc'] = R[p]['mc']['var'] > tau_u
        R[p]['risk'] = 2 * R[p]['level'] + R[p]['unc'].astype(int)
    R['val'].update(y=y_val, ymal=np.isin(y_val, MAL_IDX).astype(int), groups=GROUPS[sp['val']])
    y_test = LABELS[sp['test']]
    R['test'].update(y=y_test, ymal=np.isin(y_test, MAL_IDX).astype(int), groups=GROUPS[sp['test']], rows=sp['test'])
    for ds, d in ext.items():
        R[ds].update(y=_value(d['y'], k), ymal=_value(d['ymal'], k), groups=_value(d['groups'], k))
    R['meta'] = {'T': T, 'T_det': T_det, 'T_single': Ts, 'tau_u': tau_u, 'dmin': dmin, 'dmax': dmax}
    return R

# ---------------------------------------------------------------- metrics
def p_mal(p):
    return p[:, MAL_IDX].sum(1)

def ece(p, y, n_bins=15):
    conf, ok = p.max(1), (p.argmax(1) == y).astype(float); bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def brier(p, y):
    return float(np.mean(np.sum((p - np.eye(p.shape[1])[y]) ** 2, 1)))

def balanced_acc_present(y, p):
    """Balanced accuracy over the classes present (images outside the 7 classes, y = -1, are ignored)."""
    m = y >= 0; yy, pred = y[m], p[m].argmax(1)
    return float(np.mean([np.mean(pred[yy == c] == c) for c in sorted(np.unique(yy))]))

def cls_metrics(y, ymal, p):
    kn = y >= 0; yy, pp = y[kn], p[kn]; present = sorted(np.unique(yy)); pred = pp.argmax(1)
    return {'Accuracy': accuracy_score(yy, pred),
            'Balanced acc.': float(np.mean([np.mean(pred[yy == c] == c) for c in present])),
            'Macro F1': f1_score(yy, pred, labels=present, average='macro', zero_division=0),
            'Macro AUC': float(np.mean([roc_auc_score((yy == c).astype(int), pp[:, c]) for c in present if 0 < (yy == c).sum() < len(yy)])),
            'Malignant AUC': roc_auc_score(ymal, p_mal(p)) if 0 < ymal.sum() < len(ymal) else np.nan,
            'ECE': ece(pp, yy), 'Brier': brier(pp, yy), 'NLL': nll(pp, yy)}

CONFIGS = OrderedDict([(LABEL_OF[m], (lambda R, m=m: R['single_det'][m])) for m in MODELS] +
                      [('Ensemble (soft vote)', lambda R: R['ens_det']),
                       ('Ensemble + MC Dropout', lambda R: R['mc']['probs']),
                       ('Ensemble + MC Dropout + temperature scaling', lambda R: R['full'])])

In [ ]:
ham, LABELS, GROUPS = load_ham()
SPLITS = load_splits(); FOLDS = sorted(SPLITS)
HAM_IMGS = np.load(DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.npy', mmap_mode='r')
print(f'{len(ham)} images, {len(set(GROUPS))} lesions, folds {FOLDS}')

## 1. Cached features and framework rebuild

In [ ]:
for k in tqdm(FOLDS, desc='feature caches'):
    ensure_fold_cache(k, HAM_IMGS, LABELS, SPLITS)
FOLD = {k: build_fold(k) for k in tqdm(FOLDS, desc='rebuilding the framework')}
display(pd.DataFrame({k: {'T (framework)': FOLD[k]['meta']['T'], 'uncertainty threshold': FOLD[k]['meta']['tau_u']}
                      for k in FOLDS}).T.round(4))

## 2. Per-fold results

In [ ]:
rows = []
for k in FOLDS:
    R = FOLD[k]['test']
    for cfg, fn in CONFIGS.items():
        rows.append({'fold': k, 'configuration': cfg, **cls_metrics(R['y'], R['ymal'], fn(R))})
PER_FOLD = pd.DataFrame(rows)
PER_FOLD.to_csv(DIRS['tables'] / 'internal_per_fold.csv', index=False)
METRICS = ['Accuracy', 'Balanced acc.', 'Macro F1', 'Macro AUC', 'Malignant AUC', 'ECE', 'Brier', 'NLL']
display(PER_FOLD.groupby('configuration', sort=False)[METRICS].agg(lambda s: mean_sd(s, 4)))

### Reproduction check

If the original pipeline's per-fold table is present in the output folder, the rebuilt values are compared with it.

In [ ]:
saved = DIRS['tables'] / 'per_fold_all_configs.csv'
NAME_IN_PIPELINE = {'EfficientNet-B4': 'EfficientNet-B4', 'ConvNeXt-Base': 'ConvNeXt-Base', 'DenseNet-121': 'DenseNet-121',
                    'Ensemble (soft vote)': 'Ensemble (Focal + LS)', 'Ensemble + MC Dropout': 'Ensemble (Focal + LS) + MC',
                    'Ensemble + MC Dropout + temperature scaling': 'Full framework (Focal + LS + MC + T-scaling)'}
if saved.exists():
    pf = pd.read_csv(saved); gate = []
    for _, r in PER_FOLD.iterrows():
        s = pf[(pf['config'] == NAME_IN_PIPELINE[r['configuration']]) & (pf['fold'] == r['fold'])]
        if len(s) and 'Balanced acc.' in s:
            gate.append(abs(float(s['Balanced acc.'].iloc[0]) - r['Balanced acc.']))
    print(f'Compared {len(gate)} values with the pipeline table; max |difference| = {max(gate) if gate else float("nan"):.2e}')
else:
    print('No pipeline table found - nothing to compare (expected when the models were trained with notebook 02).')

## 3. Results table for the README

In [ ]:
show = ['EfficientNet-B4', 'ConvNeXt-Base', 'DenseNet-121', 'Ensemble (soft vote)', 'Ensemble + MC Dropout + temperature scaling']
cols = [('Accuracy', 'accuracy_pct', 100), ('Balanced acc.', 'balanced_accuracy_pct', 100), ('Macro F1', 'macro_f1_pct', 100),
        ('Macro AUC', 'macro_auc', 1), ('Malignant AUC', 'malignant_vs_benign_auc', 1), ('ECE', 'ece', 1)]
out = []
for cfg in show:
    g = PER_FOLD[PER_FOLD['configuration'] == cfg]; row = {'model': cfg}
    for src, dst, scale in cols:
        nd = 2 if scale == 100 else 3
        row[f'{dst}_mean'] = round(scale * g[src].mean(), nd); row[f'{dst}_std'] = round(scale * g[src].std(ddof=1), nd)
    out.append(row)
INTERNAL = pd.DataFrame(out); INTERNAL.to_csv(RESULTS_DIR / 'internal_results.csv', index=False)
display(INTERNAL)

## 4. Pooled out-of-fold predictions: per-class results and confusion matrix

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
n = len(LABELS); P_OOF = np.zeros((n, C))
for k in FOLDS:
    P_OOF[FOLD[k]['test']['rows']] = FOLD[k]['test']['full']
pred = P_OOF.argmax(1)
prec, rec, f1, sup = precision_recall_fscore_support(LABELS, pred, labels=range(C), zero_division=0)
per_class = pd.DataFrame({'precision': prec, 'recall': rec, 'F1': f1,
                          'one-vs-rest AUC': [roc_auc_score((LABELS == c).astype(int), P_OOF[:, c]) for c in range(C)],
                          'support': sup}, index=CLASSES).round(3)
display(per_class); per_class.to_csv(DIRS['tables'] / 'internal_per_class.csv')
print(f'Pooled balanced accuracy {balanced_accuracy_score(LABELS, pred):.4f}')

cm = confusion_matrix(LABELS, pred, labels=range(C)); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
for i in range(C):
    for j in range(C):
        ax.text(j, i, f'{cmn[i, j]:.2f}\n({cm[i, j]})', ha='center', va='center', fontsize=7,
                color='white' if cmn[i, j] > 0.55 else 'black')
ax.set_xticks(range(C)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(C)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted class'); ax.set_ylabel('true class')
ax.set_title('HAM10000, pooled out-of-fold predictions\n(row-normalised; counts in brackets)', fontsize=10)
plt.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight'); plt.show()

## 5. Calibration

In [ ]:
cal = []
for k in FOLDS:
    R = FOLD[k]['test']; y = R['y']
    for m in MODELS:
        cal.append({'fold': k, 'model': LABEL_OF[m], 'ECE before': ece(R['single_det'][m], y), 'ECE after': ece(R['single_cal'][m], y),
                    'Brier after': brier(R['single_cal'][m], y), 'NLL after': nll(R['single_cal'][m], y), 'T': FOLD[k]['meta']['T_single'][m]})
    cal.append({'fold': k, 'model': 'Ensemble + MC (framework)', 'ECE before': ece(R['mc']['probs'], y), 'ECE after': ece(R['full'], y),
                'Brier after': brier(R['full'], y), 'NLL after': nll(R['full'], y), 'T': FOLD[k]['meta']['T']})
CAL = pd.DataFrame(cal); CAL.to_csv(DIRS['tables'] / 'calibration_per_fold.csv', index=False)
display(CAL.groupby('model', sort=False)[['ECE before', 'ECE after', 'Brier after', 'NLL after', 'T']].agg(lambda s: mean_sd(s, 3)))

def reliability(ax, p, y, title, n_bins=15):
    conf, ok = p.max(1), (p.argmax(1) == y); bins = np.linspace(0, 1, n_bins + 1); mids, accs = [], []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            mids.append((bins[i] + bins[i + 1]) / 2); accs.append(ok[m].mean())
    ax.bar(mids, accs, width=1 / n_bins, edgecolor='k', alpha=.75); ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title(f'{title}\nECE {ece(p, y):.3f}', fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

P_ = {'EfficientNet-B4': np.zeros((n, C)), 'ConvNeXt-Base': np.zeros((n, C)), 'DenseNet-121': np.zeros((n, C)),
      'MC ensemble': np.zeros((n, C)), 'MC ensemble + T': np.zeros((n, C))}
for k in FOLDS:
    R = FOLD[k]['test']; r = R['rows']
    for m in MODELS:
        P_[LABEL_OF[m]][r] = R['single_det'][m]
    P_['MC ensemble'][r] = R['mc']['probs']; P_['MC ensemble + T'][r] = R['full']
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, (name, p) in zip(axes, P_.items()):
    reliability(ax, p, LABELS, name)
axes[0].set_ylabel('accuracy'); [a.set_xlabel('confidence') for a in axes]
plt.tight_layout(); plt.savefig(DIRS['figures'] / 'reliability_diagrams.png', dpi=150); plt.show()

## 6. Choice of uncertainty signal (validation data only)

Four signals are compared by how well they identify misclassified images on the **validation** partitions; the referral signal is
the one with the highest mean validation AUC.

In [ ]:
SIGNALS = OrderedDict([('1 - maximum probability', lambda R: 1 - R['full'].max(1)), ('predictive entropy', lambda R: R['mc']['entropy']),
                       ('mutual information', lambda R: R['mc']['mi']), ('MC variance (sigma^2)', lambda R: R['mc']['var'])])
sig = []
for k in FOLDS:
    for part in ('val', 'test'):
        R = FOLD[k][part]; err = (R['full'].argmax(1) != R['y']).astype(int)
        sig.append({'fold': k, 'partition': part, **{s: roc_auc_score(err, f(R)) for s, f in SIGNALS.items()}})
SIG = pd.DataFrame(sig)
sig_table = SIG.groupby('partition')[list(SIGNALS)].mean().T.round(3); display(sig_table)
print('Selected on validation data:', sig_table['val'].idxmax())
sig_table.to_csv(DIRS['tables'] / 'uncertainty_signal_selection.csv')

## Summary

* The ensemble improves on every single backbone; MC Dropout and temperature scaling keep balanced accuracy unchanged while
  providing uncertainty estimates and calibrated probabilities.
* The most frequent error type is visible in the confusion matrix (row `nv`, column `mel`).